In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
import pandas as pd
import numpy as np

In [18]:
import pandas as pd

df = pd.read_csv("df_ifrs9_prepared.csv")
baseline_er = pd.read_csv("baseline_er.csv")


In [19]:
import pandas as pd
import numpy as np

# Colonnes retenues
num_cols = ["AGE_PRET", "B_RESMAT", "Tx", "MREVTOT", "AGE_CLI", "NBIMP", "NB_ECH"]
cat_cols = ["produit", "CSP", "CLASSACT"]

cols_needed = num_cols + cat_cols + ["FLAG_ER", "ER_obs"]

df_model = df[cols_needed].copy()

# Supprimer toute ligne contenant NaN
df_model = df_model.dropna()

print("Shape after dropna:", df_model.shape)


Shape after dropna: (7564, 12)


In [20]:
df_model = pd.get_dummies(df_model, columns=cat_cols, drop_first=True)

X = df_model.drop(columns=["FLAG_ER", "ER_obs"])
y_flag = df_model["FLAG_ER"].astype(int)
y_er = df_model["ER_obs"].astype(float)


train split test 

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_flag_train, y_flag_test, y_er_train, y_er_test = train_test_split(
    X, y_flag, y_er,
    test_size=0.25,
    random_state=42,
    stratify=y_flag
)


modèle 1 - Logit occurence 


## Modélisation de l’occurrence du remboursement anticipé

Objectif : estimer la probabilité qu’un remboursement anticipé ait lieu.

Nous utilisons une régression logistique pour prédire `FLAG_ER`.

Les indicateurs de performance utilisés sont :
- l’AUC (qualité de discrimination),
- l’accuracy (taux de bonne classification).

Ce modèle permet d’identifier les déterminants de la décision de remboursement anticipé.


In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score

logit = LogisticRegression(max_iter=1000)
logit.fit(X_train, y_flag_train)

proba_test = logit.predict_proba(X_test)[:, 1]
pred_test = logit.predict(X_test)

print("AUC:", roc_auc_score(y_flag_test, proba_test))
print("Accuracy:", accuracy_score(y_flag_test, pred_test))


AUC: 0.6631570183715381
Accuracy: 0.8878900052882073


c:\Users\sarah\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


modèle 2 - intensité simple 

## Modélisation de l’intensité du remboursement anticipé

Objectif : estimer le montant relatif remboursé (`ER_obs`).

Nous utilisons une régression linéaire simple pour prédire directement `ER_obs`.

Les prédictions sont contraintes à l’intervalle [0,1] afin de respecter la nature proportionnelle de la variable.

Les performances sont mesurées par :
- le MAE (erreur absolue moyenne),
- le RMSE (racine de l’erreur quadratique moyenne).


In [23]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

linreg = LinearRegression()
linreg.fit(X_train, y_er_train)

er_pred = linreg.predict(X_test)

# Forcer dans [0,1]
er_pred = np.clip(er_pred, 0, 1)

print("MAE intensité:", mean_absolute_error(y_er_test, er_pred))
print("RMSE intensité:", np.sqrt(mean_squared_error(y_er_test, er_pred)))


MAE intensité: 0.230440975531316
RMSE intensité: 0.28252524808622687


modèle combiné 

## Construction du remboursement anticipé attendu

Objectif : combiner occurrence et intensité.

Conformément à la décomposition suivante :

E[ER | X] = P(ER > 0 | X) × E[ER | X]

Nous multiplions :
- la probabilité prédite par le modèle logistique,
- l’intensité prédite par la régression.

On obtient ainsi une estimation complète du remboursement anticipé attendu pour chaque observation.


In [24]:
er_expected = proba_test * er_pred

print("MAE total:", mean_absolute_error(y_er_test, er_expected))
print("RMSE total:", np.sqrt(mean_squared_error(y_er_test, er_expected)))


MAE total: 0.22392657557814186
RMSE total: 0.284613399970253


baseline comparaison

In [25]:
baseline_global = y_er_train.mean()
baseline_pred = np.full_like(y_er_test, baseline_global)

print("Baseline MAE:", mean_absolute_error(y_er_test, baseline_pred))
print("Baseline RMSE:", np.sqrt(mean_squared_error(y_er_test, baseline_pred)))


Baseline MAE: 0.2389696343389316
Baseline RMSE: 0.2904267682435695


## Résultats de la première modélisation

### Modèle d’occurrence (Logistic Regression)

- AUC ≈ 0.66 : pouvoir discriminant modéré mais réel.
- Le modèle capte un signal prédictif au-delà du hasard.
- L’accuracy élevée (~0.89) doit être interprétée avec prudence si la classe majoritaire est dominante.
- Les remboursements anticipés sont partiellement explicables par les variables structurelles (âge, taux, profil).

### Modèle d’intensité (Régression linéaire)

- MAE ≈ 0.23 : erreur moyenne d’environ 23 points de pourcentage.
- RMSE ≈ 0.28 : dispersion importante des intensités.
- L’intensité des remboursements anticipés reste difficile à prédire finement.
- Une partie significative du comportement dépend probablement de facteurs non observés.

### Modèle combiné (ER attendu)

- MAE total ≈ 0.224
- RMSE total ≈ 0.285
- Amélioration par rapport à la baseline globale :
  - MAE réduit d’environ 6 %
  - RMSE légèrement amélioré

### Conclusion intermédiaire

- Il existe un signal prédictif exploitable.
- La structure par âge reste un déterminant majeur.
- Le modèle améliore la projection, mais de manière modérée.
- Une baseline comportementale par âge constitue déjà une référence solide.
